In [67]:
### bagging, voting, stacking, boosting

In [ ]:
import pandas as pd

df = pd.read_csv(r'C:\Users\konta\Documents\DIV_Academy\Module2(From_29_nov)\data\cleanned.csv')

In [69]:
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,num
0,63,Male,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,0
1,67,Male,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,2
2,67,Male,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,1
3,37,Male,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,0
4,41,Female,typical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,0
...,...,...,...,...,...,...,...,...,...,...,...
913,54,Female,asymptomatic,127.0,333.0,True,st-t abnormality,154.0,False,0.0,1
914,62,Male,typical angina,130.0,139.0,False,st-t abnormality,140.0,False,0.5,0
915,55,Male,asymptomatic,122.0,223.0,True,st-t abnormality,100.0,False,0.0,2
916,58,Male,asymptomatic,130.0,385.0,True,lv hypertrophy,140.0,False,0.5,0


In [70]:
df['num'] = (df['num'] > 0).astype(int)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [72]:
X = df.drop('num', axis=1) 
y = df['num']              

In [73]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [74]:
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
cat_cols = ['sex', 'cp', 'restecg']
bool_cols = ['fbs', 'exang']

In [75]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('bool', 'passthrough', bool_cols)
])

In [76]:
bagging_model = Pipeline([
    ('prep', preprocessor),
    ('model', BaggingClassifier(
        estimator=RandomForestClassifier(),
        n_estimators=50,
        random_state=42
    ))
])

In [77]:
voting_model = Pipeline([
    ('prep', preprocessor),
    ('model', VotingClassifier([
        ('lr', LogisticRegression(max_iter=1000)),
        ('rf', RandomForestClassifier()),
        ('svc', SVC(probability=True))
    ], voting='soft'))
])

In [78]:
stacking_model = Pipeline([
    ('prep', preprocessor),
    ('model', StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier()),
            ('svc', SVC(probability=True))
        ],
        final_estimator=LogisticRegression()
    ))
])

In [79]:
boosting_model = Pipeline([
    ('prep', preprocessor),
    ('model', GradientBoostingClassifier())
])

In [80]:
bagging_model.fit(X_train, y_train)
bagging_model.score(X_test, y_test)

0.8369565217391305

In [81]:
voting_model.fit(X_train, y_train)
voting_model.score(X_test, y_test)

0.8152173913043478

In [82]:
stacking_model.fit(X_train, y_train)
stacking_model.score(X_test, y_test)

0.842391304347826

In [83]:
boosting_model.fit(X_train, y_train)
boosting_model.score(X_test, y_test)

0.8260869565217391